In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import joblib

In [16]:
# --- SIMULATE LOADING CLEANED DATA ---
# In a real scenario, 'df_cleaned' would be your fully cleaned CSV data.
# We load the raw CSV for demonstration purposes and define a basic pipeline.
df_raw = pd.read_csv('C:/Users/Sakshi/Downloads/alzheimers_disease_data_cleaned.csv')
X = df_raw.drop(['Diagnosis', 'PatientID', 'DoctorInCharge'], axis=1)
y = df_raw['Diagnosis'].astype('category').cat.codes # Convert diagnosis to numeric labels (0, 1, 2, 3...)

# Identify feature types
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Define the preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        # Handle numerical data: impute missing values (if any remain) and scale
        ('num', 
         Pipeline([
             ('imputer', SimpleImputer(strategy='median')), # Basic Imputation
             ('scaler', StandardScaler())
         ]), 
         numerical_features),
        
        # Handle categorical data: impute and one-hot encode
        ('cat', 
         Pipeline([
             ('imputer', SimpleImputer(strategy='most_frequent')), # Basic Imputation
             ('onehot', OneHotEncoder(handle_unknown='ignore')) # Convert to 0s and 1s
         ]), 
         categorical_features)
    ])

# Create the full pipeline: Preprocessor + Model
clinical_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

# Split the data
X_train_clinical, X_test_clinical, y_train_clinical, y_test_clinical = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
print("Starting Clinical Model Training...")

# Train the pipeline (this applies preprocessing and training in one step)
clinical_pipeline.fit(X_train_clinical, y_train_clinical)

print("Clinical Model Training Complete.")

# --- Save the entire trained pipeline for deployment ---
joblib.dump(clinical_pipeline, 'clinical_model_pipeline.pkl')
print("Clinical model pipeline saved as clinical_model_pipeline.pkl")


Starting Clinical Model Training...
Clinical Model Training Complete.
Clinical model pipeline saved as clinical_model_pipeline.pkl


In [18]:
# --- SIMULATE CLEANED IMAGE DATA INPUT ---
# In a real scenario, X_train_img would be a NumPy array of shape:
# (number_of_images, height, width, channels) e.g., (1000, 128, 128, 3)
# y_train_img would be a NumPy array of one-hot encoded labels.

IMG_WIDTH = 128
IMG_HEIGHT = 128
CHANNELS = 3  # Assuming RGB images; use 1 for grayscale
NUM_CLASSES = 4 # e.g., Non-Demented, Very Mild, Mild, Moderate

# Placeholder for the actual loaded data (replace this with your loading script)
X_train_img = np.random.rand(100, IMG_HEIGHT, IMG_WIDTH, CHANNELS) 
y_train_img = tf.keras.utils.to_categorical(np.random.randint(0, NUM_CLASSES, 100), num_classes=NUM_CLASSES)

def build_cnn_model():
    model = Sequential([
        # 1. Convolutional Block
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # 2. Convolutional Block
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # 3. Dense Block
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        
        # Output Layer (Softmax for multi-class classification)
        Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

cnn_model = build_cnn_model()

c:\Users\Sakshi\anaconda3\New folder\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
print("Starting CNN Model Compilation and Training...")

# Compile the model
cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy', # Used for multi-class one-hot encoded labels
                  metrics=['accuracy'])

# Print the model structure summary
cnn_model.summary()

# Train the model (use a separate validation set for monitoring performance)
history = cnn_model.fit(
    X_train_img, y_train_img,
    epochs=10, # Start with a small number, tune this later
    batch_size=32,
    validation_split=0.1 # Use 10% of training data for validation
)

print("CNN Model Training Complete.")

# --- Save the trained CNN model for deployment ---
cnn_model.save('imaging_model.h5')
print("CNN model saved as imaging_model.h5")

Starting CNN Model Compilation and Training...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,392,836 (28.20 MB)

 Trainable params: 7,392,836 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 241ms/step - accuracy: 0.2333 - loss: 6.0498 - val_accuracy: 0.4000 - val_loss: 2.4414
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.2667 - loss: 3.6390 - val_accuracy: 0.2000 - val_loss: 1.5975
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.2444 - loss: 1.9197 - val_accuracy: 0.1000 - val_loss: 1.4107
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.2667 - loss: 1.4105 - val_accuracy: 0.2000 - val_loss: 1.3890
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.2667 - loss: 1.3808 - val_accuracy: 0.1000 - val_loss: 1.3861
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.3111 - loss: 1.3834 - val_accuracy: 0.3000 - val_loss: 1.3858
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.3111 - loss: 1.3810 - val_accuracy: 0.4000 - val_loss: 1.3856
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.3333 - loss: 1.3783 - val_accuracy: 0.3000 - val_loss:

CNN Model Training Complete.
CNN model saved as imaging_model.h5
